# GemmaFit v5.1 E2B Model Generation Eval

Post-training Colab notebook. This notebook does **not** train, merge, or convert the model. It only runs real generation eval on the merged HF model:

```text
held-out evidence packet -> merged E2B generate -> JSON function call parser -> v5 gates
```

Use this to decide whether the E2B evidence router is actually ready before LiteRT conversion / Pixel smoke.

In [ ]:
# Cell 0: Mount Drive + basic paths
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import zipfile
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
RUN_NAME = os.environ.get('RUN_NAME', 'google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases')
MERGED_DIR = Path(os.environ.get('MERGED_DIR', str(DRIVE_ROOT / 'trained_outputs' / 'merged_fp16' / RUN_NAME)))
DATASET_PATH = Path(os.environ.get('DATASET_PATH', str(REPO_DIR / 'finetune' / 'data' / 'gemmafit_v5_1_e2b_evidence_router.json')))
METRICS_DIR = REPO_DIR / 'finetune' / 'metrics'
OUTPUT_JSON = Path(os.environ.get('GEN_EVAL_OUTPUT', str(METRICS_DIR / 'model_generation_eval_v5_e2b.json')))
PREDICTIONS_JSONL = Path(os.environ.get('GEN_EVAL_PREDICTIONS', str(METRICS_DIR / 'model_generation_predictions_v5_e2b.jsonl')))

print('Drive root :', DRIVE_ROOT)
print('Repo dir   :', REPO_DIR)
print('Merged HF  :', MERGED_DIR)
print('Dataset    :', DATASET_PATH)
print('Output     :', OUTPUT_JSON)
print('Predictions:', PREDICTIONS_JSONL)
print('Merged exists :', MERGED_DIR.exists())
print('Dataset exists:', DATASET_PATH.exists())


In [ ]:
# Cell 1: Clone or update repo
# If you already cloned the branch manually, this cell will just fetch/pull it.
import subprocess
import os
from pathlib import Path

REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')
BRANCH = os.environ.get('GEMMAFIT_BRANCH', 'codex/e2b-video-capability-probes')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('Repo ready:', REPO_DIR)
print('HEAD:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)


In [ ]:
# Cell 2: Install / verify runtime dependencies
# This is eval-only. It does not install Unsloth or training dependencies.
import subprocess
import sys

INSTALL_DEPS = os.environ.get('INSTALL_DEPS', '1') == '1'
if INSTALL_DEPS:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-U',
        'transformers>=5.5.0', 'accelerate', 'safetensors', 'sentencepiece'
    ], check=True)
else:
    print('INSTALL_DEPS=0; skipping pip install')

import torch
import transformers
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# Cell 3: Preflight dataset and merged model
import hashlib
import json
import subprocess
import sys
import zipfile
from pathlib import Path

TRAINING_DONE = REPO_DIR / 'finetune' / 'metrics' / 'training_done_v5_e2b.json'
EXPECTED_DATASET_SHA256 = ''
if TRAINING_DONE.exists():
    try:
        EXPECTED_DATASET_SHA256 = json.loads(TRAINING_DONE.read_text(encoding='utf-8')).get('dataset_sha256', '')
    except Exception as exc:
        print('Could not read training_done hash:', exc)

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def restore_dataset_from_drive_zip() -> bool:
    package_dir = DRIVE_ROOT / 'artifact_packages'
    if not package_dir.exists():
        return False
    packages = sorted(package_dir.glob('gemmafit-v5-1-e2b-training-metadata-*.zip'), key=lambda x: x.stat().st_mtime, reverse=True)
    for zip_path in packages:
        print('Checking package:', zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            member = 'finetune/data/gemmafit_v5_1_e2b_evidence_router.json'
            if member not in zf.namelist():
                continue
            DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, DATASET_PATH.open('wb') as dst:
                while True:
                    chunk = src.read(1024 * 1024)
                    if not chunk:
                        break
                    dst.write(chunk)
            print('Restored dataset from package:', zip_path)
            return True
    return False

def regenerate_dataset() -> None:
    print('No Drive dataset package found; regenerating deterministic v5.1 hard-case dataset.')
    cmd = [
        sys.executable,
        str(REPO_DIR / 'finetune' / 'data' / 'generate_v5_e2b_evidence_router.py'),
        '--train-count', '50000',
        '--validation-count', '6000',
        '--out', str(DATASET_PATH),
        '--validate',
        '--hard-cases',
        '--zh-tw-ratio', '0.45',
        '--schema-fuzz-ratio', '0.25',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not MERGED_DIR.exists():
    raise FileNotFoundError(f'Merged model not found: {MERGED_DIR}. Run Phase 8 merge in the training notebook first.')
if not DATASET_PATH.exists():
    print('Dataset missing from repo checkout:', DATASET_PATH)
    if not restore_dataset_from_drive_zip():
        regenerate_dataset()

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset still missing after restore/generate: {DATASET_PATH}')

actual_sha = file_sha256(DATASET_PATH)
print('Dataset file sha256:', actual_sha)
if EXPECTED_DATASET_SHA256:
    print('Expected sha256    :', EXPECTED_DATASET_SHA256)
    if actual_sha != EXPECTED_DATASET_SHA256:
        raise RuntimeError('Dataset SHA256 mismatch. Do not run generation eval against a different dataset.')

with DATASET_PATH.open('r', encoding='utf-8') as f:
    dataset = json.load(f)
validation_counts = {k: len(v) for k, v in dataset.get('validation', {}).items()}
print('Dataset version:', dataset.get('version'))
print('Train rows:', len(dataset.get('train', [])))
print('Validation rows:', validation_counts)
print('Validation total:', sum(validation_counts.values()))
print('Merged files sample:')
for p in sorted(MERGED_DIR.iterdir())[:20]:
    print(' ', p.name, p.stat().st_size if p.is_file() else '<dir>')


In [ ]:
# Cell 4: Quick dry-run row selection, no model load
# Use this to verify the evaluator script and row selection before loading the 10GB model.
import subprocess
import sys

DRY_RUN_LIMIT = int(os.environ.get('DRY_RUN_LIMIT', '12'))
cmd = [
    sys.executable, str(REPO_DIR / 'finetune' / 'eval_v5_e2b_model_generation.py'),
    '--model', str(MERGED_DIR),
    '--dataset', str(DATASET_PATH),
    '--output', str(METRICS_DIR / 'model_generation_eval_v5_e2b_dry_run.json'),
    '--limit', str(DRY_RUN_LIMIT),
    '--dry-run',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Cell 5: Real model generation eval
# Start with LIMIT=300. If it passes, rerun with LIMIT=1000.
# This can take a while because it runs real E2B generation row by row.
# The cell no longer hides subprocess details behind CalledProcessError:
# stdout/stderr are streamed to console and saved to a log file.
import json
import subprocess
import sys
from pathlib import Path

LIMIT = int(os.environ.get('GEN_EVAL_LIMIT', '300'))
# Default non-strict so the notebook continues and Cell 6 can show gate failures.
# Set GEN_EVAL_STRICT=1 when you want the command itself to return non-zero on failed gates.
STRICT = os.environ.get('GEN_EVAL_STRICT', '0') == '1'
MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '256'))
DTYPE = os.environ.get('GEN_EVAL_DTYPE', 'float16')
ROW_TYPES = os.environ.get('GEN_EVAL_ROW_TYPES', 'all')
SPLITS = os.environ.get('GEN_EVAL_SPLITS', 'all')
LOG_PATH = Path(os.environ.get('GEN_EVAL_LOG', str(METRICS_DIR / 'model_generation_eval_v5_e2b.log')))

cmd = [
    sys.executable, str(REPO_DIR / 'finetune' / 'eval_v5_e2b_model_generation.py'),
    '--model', str(MERGED_DIR),
    '--dataset', str(DATASET_PATH),
    '--output', str(OUTPUT_JSON),
    '--predictions-output', str(PREDICTIONS_JSONL),
    '--limit', str(LIMIT),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--dtype', DTYPE,
    '--row-types', ROW_TYPES,
    '--splits', SPLITS,
]
if STRICT:
    cmd.append('--strict')

print('Running:', ' '.join(cmd))
print('Log path:', LOG_PATH)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
    rc = proc.wait()

print('Generation eval return code:', rc)
if rc != 0:
    if OUTPUT_JSON.exists():
        report = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
        print('Eval produced a report but returned non-zero. This usually means strict gate failures, not a crashed run.')
        print('Gate failures:', report.get('gate_failures'))
        print('Summary:', json.dumps(report.get('summary', {}), indent=2, ensure_ascii=False))
    else:
        tail = LOG_PATH.read_text(encoding='utf-8', errors='replace')[-5000:] if LOG_PATH.exists() else ''
        raise RuntimeError(f'Generation eval crashed before writing {OUTPUT_JSON}. Log tail:\n{tail}')
else:
    print('Generation eval command completed successfully.')


In [ ]:
# Cell 6: Inspect eval result
import json
from pathlib import Path

report = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
print(json.dumps({
    'summary': report.get('summary'),
    'generation': report.get('generation'),
    'gate_failures': report.get('gate_failures'),
    'row_type_summary': report.get('row_type_summary'),
}, indent=2, ensure_ascii=False))

failures = report.get('failure_samples', [])
print('Failure sample count:', len(failures))
for item in failures[:5]:
    print(json.dumps(item, indent=2, ensure_ascii=False)[:3000])


In [ ]:
# Cell 7: Package generation eval artifacts to Drive
import json
import time
import zipfile
from pathlib import Path

PACKAGE_DIR = DRIVE_ROOT / 'artifact_packages'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
ZIP_PATH = PACKAGE_DIR / f'gemmafit-v5-e2b-model-generation-eval-{STAMP}.zip'
LOG_PATH = Path(os.environ.get('GEN_EVAL_LOG', str(METRICS_DIR / 'model_generation_eval_v5_e2b.log')))

files = [
    (OUTPUT_JSON, 'finetune/metrics/model_generation_eval_v5_e2b.json'),
    (PREDICTIONS_JSONL, 'finetune/metrics/model_generation_predictions_v5_e2b.jsonl'),
    (LOG_PATH, 'finetune/metrics/model_generation_eval_v5_e2b.log'),
    (METRICS_DIR / 'model_generation_eval_v5_e2b_dry_run.json', 'finetune/metrics/model_generation_eval_v5_e2b_dry_run.json'),
]

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path, arcname in files:
        if Path(path).exists():
            zf.write(path, arcname)
            print('added:', arcname)
        else:
            print('missing:', path)

print('Package written:', ZIP_PATH)
print('Size bytes:', ZIP_PATH.stat().st_size)


## Notes

Readiness meaning:

- If Cell 5 passes with `--strict`, the merged HF model has passed real generation eval for the selected sample size.
- This still does not mean Android-ready. Android-ready requires LiteRT conversion + Pixel LiteRT smoke.
- This evaluates evidence-router behavior only. It does not evaluate raw video understanding.